# Model 1: Multinomial Logistic Regression
## IoT Saldırı Tipi Sınıflandırması (Multi-class)

Bu notebook, **Edge-IIoTset** veri seti üzerinde **Attack_type** kolonunu hedef değişken olarak kullanan
multinomial Logistic Regression modelini adım adım eğitir ve değerlendirir.

### Neden Logistic Regression?
- **Basit ve yorumlanabilir**: Katsayılar doğrudan feature etkisini gösterir
- **Hızlı eğitim**: Büyük veri setlerinde bile makul sürede eğitilir
- **Baseline model**: Diğer modellerin karşılaştırılacağı referans çizgisi
- **Multinomial destek**: Spark MLlib `family="multinomial"` ile çok sınıflı problemi doğrudan çözer

### Pipeline Adımları
1. Gold Delta Lake → Veri yükleme
2. Feature vektörleme + StringIndexer (Attack_type → label)
3. Sınıf ağırlıkları (class imbalance)
4. Stratified Train/Test split (%80/%20)
5. StandardScaler + Multinomial LR
6. CrossValidator ile hiperparametre arama (regParam, elasticNetParam)
7. Test seti değerlendirme (Accuracy, F1, Precision, Recall)
8. Confusion matrix ve feature importance
9. MLflow loglama

## 1. Kütüphaneler ve Spark Session

In [1]:
import sys
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import StandardScaler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

sys.path.insert(0, "/opt/bitnami/spark")

from spark.spark_session import get_spark
from ml.utils import (
    run_ml_pipeline_multiclass,
    evaluate_model_multiclass,
    compute_confusion_matrix_multiclass,
    log_to_mlflow,
)

print("Kütüphaneler yüklendi.")

Kütüphaneler yüklendi.


In [2]:
spark = get_spark("Notebook-LogisticRegression-Multiclass")
print(f"Spark version: {spark.version}")
print(f"App name: {spark.sparkContext.appName}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/13 15:50:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.4.2
App name: Notebook-LogisticRegression-Multiclass


## 2. Veri Yükleme ve Feature Hazırlığı

Gold katmanından ML'e hazır veriyi yüklüyoruz. `run_ml_pipeline_multiclass` fonksiyonu şu adımları otomatik yapar:
- Gold Delta Lake'ten batch okuma
- Numerik feature seçimi
- StringIndexer ile Attack_type → label dönüşümü
- VectorAssembler ile feature vektörü
- Sınıf ağırlıkları (balanced formula)
- Stratified train/test split

In [4]:
SAMPLE_SIZE = 30000  # Hızlı test için örneklem boyutu (None = tüm veri)

stage_start = time.perf_counter()
train_df, test_df, feature_cols, label_index_model = run_ml_pipeline_multiclass(
    spark,
    split_log_stats=True,
)
label_names = list(label_index_model.labels)
num_classes = len(label_names)

print(f"\nPipeline süresi: {time.perf_counter() - stage_start:.2f}s")
print(f"Feature sayısı: {len(feature_cols)}")
print(f"Sınıf sayısı: {num_classes}")
print(f"Sınıflar: {label_names}")


🧱 Multi-class ML pipeline başlatıldı...
✅ MLflow bağlantısı kuruldu: http://mlflow-server:5000
   Experiment: iot_intrusion_detection
   ⏱️ MLflow: 0.63s
📥 Gold katmanından veri okunuyor: /opt/bitnami/spark/delta-storage/gold/ml_ready
   ✅ Okuma tamamlandı. Kolon sayısı: 65
   ⏱️ Gold okuma (lazy): 0.09s
   📊 42 numerik feature kolon seçildi.
   ⏳ [1] prepare_features_multiclass (lazy)...
🔧 Multi-class feature hazırlığı: 42 feature, label='Attack_type'


   ✅ Multi-class hazırlık tamam (sınıf sayısı: 15).
   📚 Sınıf eşlemesi (label index → orijinal isim):
       0 → Normal
       1 → DDoS_UDP
       2 → DDoS_ICMP
       3 → Ransomware
       4 → DDoS_HTTP
       5 → SQL_injection
       6 → Uploading
       7 → DDoS_TCP
       8 → Backdoor
       9 → Vulnerability_scanner
      10 → Port_Scanning
      11 → XSS
      12 → Password
      13 → MITM
      14 → Fingerprinting
   ✅ [1] prepare_features_multiclass: 2.01s
   ⏳ [2] Tek seferlik MATERIALIZE (coalesce + persist + count)...


[Stage 50:===========================================>              (6 + 2) / 8]

   ✅ [2] Materialize tamam: 157,800 satır, 2.32s
   ⏳ [3] Multi-class class weights...


⚖️  Multi-class sınıf ağırlıkları (n=15):
   label= 0  weight=0.4329
   label= 1  weight=0.7256
   label= 2  weight=0.7466
   label= 3  weight=0.9629
   label= 4  weight=0.9961
   label= 5  weight=1.0203
   label= 6  weight=1.0244
   label= 7  weight=1.0266
   label= 8  weight=1.0319
   label= 9  weight=1.0441
   label=10  weight=1.0446
   label=11  weight=1.0466
   label=12  weight=1.0532
   label=13  weight=8.6656
   label=14  weight=10.5095
   ✅ [3] Weights: 0.24s
   ⏳ [4] randomSplit (cached'ten)...
   ✅ [4] Train: 126,322 | Test: 31,478 (1.61s)

   Sınıf dağılımı (train+test):
    0  Normal                          24,301
    1  DDoS_UDP                        14,498
    2  DDoS_ICMP                       14,090
    3  Ransomware                      10,925
    4  DDoS_HTTP                       10,561
    5  SQL_injection                   10,311
    6  Uploading                       10,269
    7  DDoS_TCP                        10,247
    8  Backdoor                        10,1

In [5]:
# Train ve Test setlerinin boyutları
train_count = train_df.count()
test_count = test_df.count()
print(f"Train: {train_count:,} satır")
print(f"Test:  {test_count:,} satır")
print(f"Toplam: {train_count + test_count:,} satır")

# Sınıf dağılımı
print("\nTrain seti sınıf dağılımı:")
train_df.groupBy("label").count().orderBy("label").show()

Train: 126,322 satır
Test:  31,478 satır
Toplam: 157,800 satır

Train seti sınıf dağılımı:
+-----+-----+
|label|count|
+-----+-----+
|  0.0|19451|
|  1.0|11540|
|  2.0|11315|
|  3.0| 8728|
|  4.0| 8468|
|  5.0| 8264|
|  6.0| 8296|
|  7.0| 8231|
|  8.0| 8108|
|  9.0| 8049|
| 10.0| 8095|
| 11.0| 8004|
| 12.0| 7966|
| 13.0|  982|
| 14.0|  825|
+-----+-----+



## 3. Model Pipeline Oluşturma

Logistic Regression pipeline'ı iki aşamadan oluşur:

1. **StandardScaler**: Feature'ları normalize eder (mean=0, std=1). LR katsayılarının doğru hesaplanması için kritik.
2. **LogisticRegression**: Multinomial mod ile çok sınıflı sınıflandırma. `classWeight` kolonu ile sınıf dengesizliği telafi edilir.

### Hiperparametreler
| Parametre | Açıklama | Aranacak Değerler |
|-----------|----------|------------------|
| `regParam` | L1/L2 regularizasyon gücü | 0.01, 0.1 |
| `elasticNetParam` | L1/L2 karışım oranı (0=L2, 1=L1) | 0.0, 0.5 |
| `maxIter` | Maksimum iterasyon | 100 |

In [6]:
# StandardScaler + LogisticRegression pipeline
scaler = StandardScaler(
    inputCol="features", outputCol="scaled_features",
    withMean=False, withStd=True,
)

lr = LogisticRegression(
    featuresCol="scaled_features",
    labelCol="label",
    weightCol="classWeight",
    family="multinomial",
    maxIter=100,
    regParam=0.01,
)

pipeline = Pipeline(stages=[scaler, lr])
print("Pipeline oluşturuldu: StandardScaler → LogisticRegression (multinomial)")

Pipeline oluşturuldu: StandardScaler → LogisticRegression (multinomial)


## 4. Cross Validation ile Hiperparametre Arama

In [7]:
# Hiperparametre grid'i
reg_params = [0.01, 0.1]
elastic_net_params = [0.0, 0.5]
num_folds = 2

param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, reg_params)
    .addGrid(lr.elasticNetParam, elastic_net_params)
    .build()
)

cv_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1",
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=cv_evaluator,
    numFolds=num_folds,
    seed=42,
    parallelism=2,
)

total_cv_runs = len(param_grid) * num_folds
print(f"Grid boyutu: {len(param_grid)} kombinasyon")
print(f"Fold sayısı: {num_folds}")
print(f"Toplam fit: {total_cv_runs}")

Grid boyutu: 4 kombinasyon
Fold sayısı: 2
Toplam fit: 8


## 5. Model Eğitimi

In [ ]:
print("Cross Validation başlatılıyor...")
cv_start = time.perf_counter()
cv_model = cv.fit(train_df)
cv_duration = time.perf_counter() - cv_start
print(f"CV tamamlandı! Süre: {cv_duration:.2f}s")

# En iyi model parametreleri
best_pipeline_model = cv_model.bestModel
best_lr_model = best_pipeline_model.stages[-1]
print(f"\nEn iyi parametreler:")
print(f"  regParam:        {best_lr_model.getRegParam()}")
print(f"  elasticNetParam: {best_lr_model.getElasticNetParam()}")
print(f"  maxIter:         {best_lr_model.getMaxIter()}")

Cross Validation başlatılıyor...


26/05/13 15:51:02 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/13 15:51:02 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


In [ ]:
# CV sonuçlarını incele
print("CV Sonuçları (F1-Score):")
print(f"{'#':<4} {'regParam':<12} {'elasticNet':<12} {'Avg F1':>10}")
print("-" * 40)
for i, (params, score) in enumerate(zip(param_grid, cv_model.avgMetrics)):
    param_str = {p.name: v for p, v in params.items()}
    marker = " ← best" if score == max(cv_model.avgMetrics) else ""
    print(f"{i+1:<4} {param_str.get('regParam', ''):<12} {param_str.get('elasticNetParam', ''):<12} {score:>10.4f}{marker}")

## 6. Test Seti Değerlendirme

In [ ]:
# Test seti üzerinde tahmin
predictions = best_pipeline_model.transform(test_df)

# Metrikleri hesapla
metrics = evaluate_model_multiclass(predictions, num_classes=num_classes)

In [ ]:
# Confusion Matrix
confusion = compute_confusion_matrix_multiclass(predictions, label_names=label_names)

In [ ]:
# Metrik özet tablosu
print("\n" + "=" * 40)
print("SONUÇ ÖZETİ")
print("=" * 40)
print(f"Accuracy:  {metrics.get('accuracy', 0):.4f}")
print(f"F1-Score:  {metrics.get('f1_score', 0):.4f}")
print(f"Precision: {metrics.get('precision', 0):.4f}")
print(f"Recall:    {metrics.get('recall', 0):.4f}")

## 7. Feature Importance

Multinomial LR'de `coefficientMatrix` (numClasses × numFeatures) boyutundadır.
Her feature için sınıflar arası **|katsayı| ortalaması** = global önem skoru olarak kullanılır.

In [ ]:
# Feature importance hesapla
coef_matrix = best_lr_model.coefficientMatrix.toArray()
abs_matrix = np.abs(coef_matrix)

# Global importance: sınıflar arası ortalama |katsayı|
global_importance = np.mean(abs_matrix, axis=0)
fi_pairs = sorted(zip(feature_cols, global_importance), key=lambda x: x[1], reverse=True)
top_features = fi_pairs[:15]

print("Top 15 Feature (sınıflar arası ortalama |katsayı|):")
print(f"{'#':<4} {'Feature':<35} {'Importance':>12}")
print("-" * 55)
for idx, (fname, importance) in enumerate(top_features, start=1):
    print(f"{idx:<4} {fname:<35} {importance:>12.6f}")

In [ ]:
# Feature Importance Görselleştirmesi
top10 = fi_pairs[:10]
names = [f[0] for f in reversed(top10)]
values = [f[1] for f in reversed(top10)]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(names, values, color="#1976D2", edgecolor="#0D47A1", height=0.6)
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + max(values) * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=9, fontweight="bold")

ax.set_xlabel("Mean |Coefficient| Across Classes", fontsize=11)
ax.set_title("Logistic Regression — Top 10 Feature Importance (Multi-class)",
             fontsize=13, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 8. Sınıf Bazında Katsayı Analizi

In [ ]:
# Her sınıf için en etkili 5 feature
for i in range(min(num_classes, 5)):  # İlk 5 sınıf
    cls_name = label_names[i] if i < len(label_names) else f"class_{i}"
    pairs = sorted(zip(feature_cols, abs_matrix[i]), key=lambda x: x[1], reverse=True)[:5]
    print(f"\nSınıf '{cls_name}' — Top 5 Feature:")
    for j, (fname, val) in enumerate(pairs, 1):
        print(f"  {j}. {fname:<30} |coef|={val:.6f}")

## 9. MLflow'a Loglama

In [ ]:
# MLflow'a log
best_params = {
    "maxIter": int(best_lr_model.getMaxIter()),
    "regParam": float(best_lr_model.getRegParam()),
    "elasticNetParam": float(best_lr_model.getElasticNetParam()),
    "family": "multinomial",
    "numClasses": int(num_classes),
    "numFolds": num_folds,
    "grid_size": len(param_grid),
    "cv_total_fits": total_cv_runs,
    "weightCol": "classWeight",
    "label_column": "Attack_type",
}

confusion_metrics = {}
for i, name in enumerate(label_names):
    confusion_metrics[f"row_total_class_{i}"] = confusion["row_totals"][i]
    confusion_metrics[f"per_class_acc_{i}"] = confusion["per_class_acc"][i]

run_id = log_to_mlflow(
    run_name="logistic_regression_notebook",
    model_type="LogisticRegression",
    params=best_params,
    metrics={**metrics, **confusion_metrics},
    model=best_pipeline_model,
    feature_importance=top_features[:10],
    tags={
        "source": "notebook",
        "model_index": "1",
        "classification_type": "multiclass",
    },
)
print(f"\nMLflow Run ID: {run_id}")

## 10. Eğitim Süreci Analizi (Objective History)

In [ ]:
# Eğitim sırasındaki objective (loss) değişimi
try:
    summary = best_lr_model.summary
    history = list(summary.objectiveHistory)
    
    if history:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(range(len(history)), history, marker='o', markersize=3, linewidth=1.5, color='#1976D2')
        ax.set_xlabel('İterasyon', fontsize=11)
        ax.set_ylabel('Objective (Loss)', fontsize=11)
        ax.set_title('Logistic Regression — Eğitim Loss Eğrisi', fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.show()
        print(f"Başlangıç loss: {history[0]:.6f}")
        print(f"Final loss:     {history[-1]:.6f}")
        print(f"Toplam iterasyon: {len(history)}")
except Exception as e:
    print(f"Objective history alınamadı: {e}")

## Özet

Bu notebook'ta **Multinomial Logistic Regression** modelini başarıyla eğittik:

- **StandardScaler** ile feature normalizasyonu uygulandı
- **CrossValidator** ile regParam ve elasticNetParam optimize edildi
- **classWeight** ile sınıf dengesizliği telafi edildi
- Sonuçlar **MLflow**'a loglandı
- Katsayı bazlı **feature importance** analizi yapıldı

Logistic Regression, basit ve hızlı bir baseline olarak diğer modellerin karşılaştırılması için referans noktası sağlar.

In [ ]:
spark.stop()
print("Spark oturumu kapatıldı.")